In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# previous work

In [ ]:
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
aseg_path = os.path.join(subj_dir, "stats/aseg.stats")
metrics = ["NVoxels", "Volume_mm3", "normMean", "normStdDev", "normMin", "normMax", "normRange"]

#read in data
aseg_data = pd.read_csv(
    aseg_path,
    sep='\s+',
    comment="#"
)

#drop the first 3 columns (as these contain no relevant info)
aseg_data = aseg_data.drop(columns = aseg_data.iloc[:, range(2)])
aseg_data.columns = [
    "NVoxels", "Volume_mm3", "StructName",
    "normMean", "normStdDev", "normMin", "normMax", "normRange"
]

#the columns are now:
#NVoxels Volume_mm3 StructName normMean normStdDev normMin normMax normRange

#pivot data with the structname
pivoted_data = aseg_data.pivot_table(
    index= "StructName",
    values= metrics
)

#transpose data
pivoted_data = pivoted_data.T

#add subject id
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])

print(pivoted_data)
print(pivoted_data.shape)

In [ ]:
#code to extract the #measure metrics for one participant
#should probably include this in the rh_ code but can also extract is separately and then merge the panda's dataframes
#now works but need to loop over all subjects for it to actually work
import pandas as pd
import re

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
aseg_path = os.path.join(subj_dir, "stats/aseg.stats")


target_metrics = [
    "BrainSegVol",
    "BrainSegVolNotVent",
    "BrainSegVolNotVentSurf",
    "VentricleChoroidVol",
    "lhCortexVol",
    "rhCortexVol",
    "CortexVol",
    "lhCerebralWhiteMatterVol",
    "rhCerebralWhiteMatterVol",
    "CerebralWhiteMatterVol",
    "SubCortGrayVol",
    "TotalGrayVol",
    "SupraTentorialVol",
    "SupraTentorialVolNotVent",
    "SupraTentorialVolNotVentVox",
    "MaskVol",
    "BrainSegVol-to-eTIV",
    "MaskVol-to-eTIV",
    "lhSurfaceHoles",
    "rhSurfaceHoles",
    "SurfaceHoles",
    "eTIV"#,
    #"avg_thickness"
]

metrics_dict = {}
with open(aseg_path, "r") as file:
    for line in file:
        if line.startswith("# Measure"):
            match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+),", line)
            #match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+|\.d+),", line)

            if match:
                metric_name = match.group(2)
                metric_value = match.group(3)
                if metric_name in target_metrics:
                    metrics_dict[metric_name] = metric_value

metrics_df = pd.DataFrame([metrics_dict])

subject_id = subj_dir.split("/")[-1]
metrics_df["SubjectID"] = subject_id

metrics_df = metrics_df[["SubjectID"] + [m for m in target_metrics if m in metrics_dict]]
metrics_df = metrics_df.set_index("SubjectID")
print(metrics_df)
print(metrics_df.shape)


In [ ]:
#code to extract the # Measure from aseg.stats and put into a dataframe
import os
import re
import pandas as pd

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_metrics = pd.DataFrame()

target_metrics = [
        "BrainSegVol",
        "BrainSegVolNotVent",
        "BrainSegVolNotVentSurf",
        "VentricleChoroidVol",
        "lhCortexVol",
        "rhCortexVol",
        "CortexVol",
        "lhCerebralWhiteMatterVol",
        "rhCerebralWhiteMatterVol",
        "CerebralWhiteMatterVol",
        "SubCortGrayVol",
        "TotalGrayVol",
        "SupraTentorialVol",
        "SupraTentorialVolNotVent",
        "SupraTentorialVolNotVentVox",
        "MaskVol",
        "BrainSegVol-to-eTIV",
        "MaskVol-to-eTIV",
        "lhSurfaceHoles",
        "rhSurfaceHoles",
        "SurfaceHoles",
        "eTIV"#,
        #"avg_thickness"
        ]

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    aseg_path = os.path.join(subj_dir, "stats", "aseg.stats")

    if os.path.exists(aseg_path):
        metrics_dict = {}
        with open(aseg_path, "r") as file:
            for line in file:
                if line.startswith("# Measure"):
                    match = re.match(r"# Measure (\w+), ([\w\-\.]+), .*?, (\d+\.\d+|\d+),", line)

                    if match:
                        metric_name = match.group(2)
                        metric_value = match.group(3)
                        if metric_name in target_metrics:
                            metrics_dict[metric_name] = metric_value

        metrics_df = pd.DataFrame([metrics_dict])

        subject_id = subj_dir.split("/")[-1]
        metrics_df["SubjectID"] = subject_id

        metrics_df = metrics_df[["SubjectID"] + [m for m in target_metrics if m in metrics_dict]]
        all_metrics = pd.concat([all_metrics, metrics_df], ignore_index=True)

final_df = all_metrics.set_index("SubjectID")

print("\nFinal combined DataFrame:")
print(final_df.tail())
#final_df.to_csv("measures_aseg_stats.csv")

In [ ]:
print("\nFinal combined DataFrame:")
print(final_df.head())
print(final_df.shape)

In [ ]:
#aseg code to extract the columns of dataframe [already used and extracted]
import os
import re
import pandas as pd

root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = []

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    aseg_path = os.path.join(subj_dir, "stats", "aseg.stats")

    if os.path.exists(aseg_path):
        aseg_data = pd.read_csv(aseg_path, sep=r"\s+", comment="#", header=None)
        aseg_data = aseg_data.drop(aseg_data.columns[:2], axis=1)

        aseg_data.columns = [
            "NVoxels", "Volume_mm3", "StructName",
            "normMean", "normStdDev", "normMin", "normMax", "normRange"
        ]

        pivoted_data = aseg_data.pivot_table(
            index=None,
            columns="StructName",
            values= ["normMean", "normStdDev"] #, "normMin", "normMax", "normRange"]
        )

        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        all_data.append(pivoted_data)

final_df = pd.concat(all_data, axis=0)

print("\nFinal combined DataFrame:")
print(final_df.head())

#final_df.to_csv("combined_aseg_stats.csv")


In [ ]:
print(final_df.shape)

(75900, 47)


final_df.to_csv("combined_aseg_stats.csv")

# Now let's create the dataset for the right and left hand side:


/project_cephfs/3022017.06/UKB/freesurfer/1533138/stats/lh.aparc.a2009s.stats
/project_cephfs/3022017.06/UKB/freesurfer/1533138/stats/rh.aparc.a2009s.stats



ColHeaders StructName NumVert SurfArea GrayVol ThickAvg ThickStd MeanCurv GausCurv FoldInd CurvInd
G&S_frontomargin                          964    697   2432  2.884 0.737     0.165     0.052       16     2.5


In [ ]:
#LH_ file for 1 subject
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
lh_path = os.path.join(subj_dir, "stats","lh.aparc.a2009s.stats")
metrics = ["ColHeaders", "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

#read in data
lh_data = pd.read_csv(
    lh_path,
    sep='\s+',
    comment="#"
)

lh_data.columns = [
    "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"
]

#pivot data with the structname
pivoted_data = lh_data.pivot_table(
    index= "StructName",
    values= ["ThickAvg"] #,"ThickStd"]
)

#transpose data
pivoted_data = pivoted_data.T
pivoted_data = pivoted_data.add_prefix("lh_")
pivoted_data = pivoted_data.add_suffix("_thickness")

#add subject id
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])
pivoted_data["SubjectID"] = subject_id

pivoted_data = pivoted_data.set_index("SubjectID")
print(pivoted_data)

In [ ]:
#for rh for 1 subj
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

#make loop here for all subj
subj_dir = os.path.join(root_dir, "1533138")
rh_path = os.path.join(subj_dir, "stats","rh.aparc.a2009s.stats")
metrics = ["ColHeaders", "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

#read in data
rh_data = pd.read_csv(
    rh_path,
    sep='\s+',
    comment="#"
)

rh_data.columns = [
    "StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"
]

#pivot data with the structname
pivoted_data = rh_data.pivot_table(
    index= "StructName",
    values= ["ThickAvg","ThickStd"]
)

#transpose data
pivoted_data = pivoted_data.T
pivoted_data = pivoted_data.add_prefix("rh_")
pivoted_data = pivoted_data.add_suffix("_thickness")


#add subject id
subject_id = os.path.basename(os.path.normpath(subj_dir))
pivoted_data.columns = pd.MultiIndex.from_product([[subject_id], pivoted_data.columns])

print(pivoted_data)
print(pivoted_data.shape)

In [ ]:
#loop over all LH files and create CSV file
import os
import re
import pandas as pd
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = pd.DataFrame()

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    lh_path = os.path.join(subj_dir, "stats", "lh.aparc.a2009s.stats")

    if os.path.exists(lh_path):
        lh_data = pd.read_csv(lh_path, sep=r"\s+", comment="#", header=None)
        lh_data.columns = ["StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

        pivoted_data = lh_data.pivot_table(
            index="StructName",
            columns=None,
            values= ["ThickAvg"] #,"ThickStd"]
        )

        #transpose data
        pivoted_data = pivoted_data.T
        pivoted_data = pivoted_data.add_prefix("lh_")
        pivoted_data = pivoted_data.add_suffix("_thickness")

        #set subj as index
        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        #add to dataframes
        all_data = pd.concat([all_data, pivoted_data])


print("\nFinal combined DataFrame:")
print(all_data.head())
print(all_data.shape)

#all_data.to_csv("lh_stats.csv")


In [ ]:
all_data.to_csv("lh_stats.csv")

In [ ]:
#loop over RH files and create csv file
import os
import re
import pandas as pd
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"

all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = pd.DataFrame()

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    rh_path = os.path.join(subj_dir, "stats", "rh.aparc.a2009s.stats")

    if os.path.exists(rh_path):
        rh_data = pd.read_csv(rh_path, sep=r"\s+", comment="#", header=None)
        rh_data.columns = ["StructName", "NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]

        pivoted_data = rh_data.pivot_table(
            index="StructName",
            columns=None,
            values= ["ThickAvg"] #,"ThickStd"]
        )

        #transpose data
        pivoted_data = pivoted_data.T
        pivoted_data = pivoted_data.add_prefix("rh_")
        pivoted_data = pivoted_data.add_suffix("_thickness")

        #set subj as index
        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        #add to dataframes
        all_data = pd.concat([all_data, pivoted_data])


print("\nFinal combined DataFrame:")
print(all_data.head())
print(all_data.shape)

#all_data.to_csv("rh_stats.csv")


all_data.to_csv("rh_stats.csv")

# Now we have the CSV files for LH, RH, Aseg.stats and Aseg measures

What i now need to do is:
1. drop the columns that do not contain any numbers (drop nan or something)
2. merge the dataframes with the "SubjectID" as a key --> google how to do this
4. merge this large datatable with the basic_demographics.csv
5. find site location in the data and add this to the large dataframe (with the SubjectID as a key)
6. compute euler (do more research and study the code given to you)
7. concat everything and you have the datatable

In [ ]:
#54 is assessment centre
#53 is date of attending
# df_misc = pd.read_csv("/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv", nrows = 10000)
# cols = list(df_misc.columns)
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(13, len(cols))])
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(1,6)])
# print(df_misc)

filepath = "/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv"
chunksize = 10000
df_first = pd.read_csv(filepath, nrows=chunksize)

#strip irrelevant cols
cols = list(df_first.columns)
df_first = df_first.drop(columns=df_first.iloc[:, range(13, len(cols))])
df_first = df_first.drop(columns=df_first.iloc[:, range(1, 6)])
keep_cols = df_first.columns.tolist()

dfs = []
for chunk in pd.read_csv(filepath, chunksize=10000):
    df_clean = chunk[keep_cols]
    dfs.append(df_clean)
df_full = pd.concat(dfs, ignore_index=True)



In [ ]:
df_full = df_full.rename(columns={"eid" : "SubjectID",
                                  "53-0.0" : "Year_scan_initial",
                                  "53-1.0" : "Year_scan_1_repeat",
                                  "53-2.0" : "Year_scan_imaging_visit",
                                  "54-0.0" : "Assessment_centre_initial",
                                  "54-2.0" : "Assessment_centre_1_repeat",
                                  "55-0.0" : "Month_scan_initial",
                                  "55-1.0" : "Month_scan_1_repeat",
                                 })

df_full = df_full.set_index("SubjectID")
#df_full.to_csv("Yr_mnth_site_scan.csv")

In [ ]:
#54 is assessment centre
#53 is date of attending
# df_misc = pd.read_csv("/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv", nrows = 10000)
# cols = list(df_misc.columns)
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(13, len(cols))])
# df_misc = df_misc.drop(columns = df_misc.iloc[:, range(1,6)])
# print(df_misc)

filepath = "/project_cephfs/3022017.05/phenotypes/current/99_miscellaneous.csv"
chunksize = 10000
df_first = pd.read_csv(filepath, nrows=chunksize)

#strip irrelevant cols
cols = list(df_first.columns)
df_first = df_first.drop(columns=df_first.iloc[:, range(13, len(cols))])
df_first = df_first.drop(columns=df_first.iloc[:, range(1, 6)])
keep_cols = df_first.columns.tolist()

dfs = []
for chunk in pd.read_csv(filepath, chunksize=10000):
    df_clean = chunk[keep_cols]
    dfs.append(df_clean)
df_full = pd.concat(dfs, ignore_index=True)



In [ ]:
#basic demographics
#31 : sex (0: female, 1 : male)
#34 : birth year

df_basic_demo = pd.read_csv("/project_cephfs/3022017.05/phenotypes/current/01_basic_demographics.csv")
df_basic_demo = df_basic_demo.drop(columns = df_basic_demo.iloc[:, range(3,4)])
df_basic_demo = df_basic_demo.rename(columns={"eid" : "SubjectID",
                                              "31-0.0" : "Sex",
                                              "34-0.0" : "Birthyear"
                                             })
df_basic_demo = df_basic_demo.set_index("SubjectID")
df_basic_demo
#df_basic_demo.to_csv("Sex_birthyear.csv")

In [ ]:
#check if specific participant is in the dataset
df_basic_demo.query('SubjectID == 1533138')

# Now we have the csv files that we need to calculate the age on the scan
What i now need to do is:
1. calculate the age of each participant during the scan
2. turn the year of scan into an integer
3. do year_scan - birthyear = age
How should i deal with the ages of the columns of the second scan? im not sure

In [ ]:
import pandas as pd
import os
import glob
import numpy as np
import re

In [ ]:
#lets make the age column at first scan
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
yr_mth_scan = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
sex_birth = os.path.join(csv_path, "Sex_birthyear.csv")


df_yr_mnth = pd.read_csv(yr_mth_scan).set_index("SubjectID")
df_demo = pd.read_csv(sex_birth).set_index("SubjectID")


In [ ]:
#turn to integer values
df_yr_mnth = df_yr_mnth.fillna(-1)
df_yr_mnth = df_yr_mnth.select_dtypes(include=['float']).astype(int)
df_yr_mnth

In [ ]:
#turn basic demo into ints
df_demo = df_demo.fillna(-1)
df_demo = df_demo.select_dtypes(include=['float']).astype(int)
df_demo

In [ ]:
df_merg = pd.merge(df_yr_mnth, df_demo, on="SubjectID")
df_merg["Age_during_scan"] = df_merg["Year_scan_initial"] - df_merg["Birthyear"]
df_merg = df_merg.fillna(-1)
df_merg["Age_during_2nd_scan"] = df_merg["Year_scan_imaging_visit"] - df_merg["Birthyear"]
df_merg




In [ ]:
df_merg.query('SubjectID == 2420905')

In [ ]:
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
df = os.path.join(csv_path, "lh_stats.csv")
df = pd.read_csv(df)

mask = ~df["SubjectID"].astype(str).str.fullmatch(r"\d+")
non_mask = ~mask
amount = mask.sum()
print(amount)

#mask = ~df["SubjectID"].astype(str).str.fullmatch(r"\d+")
#mask_other = df["SubjectID"].astype(str).str.contains(r'\d+_(?!long$|scan2$)')

#turns out there are 4008 scans that are either _long, _scan2
df.query("SubjectID == '4299969' or SubjectID == '4299969_long' or SubjectID == '4299969_scan2'")
df.dropna(axis=1)

# any file that is _scan2 or _long uses the column "Year_scan_imaging_visit"
still need to drop those files in the original dataset

In [ ]:
df_merg["Age_during_scan"] = df_merg["Year_scan_initial"] - df_merg["Birthyear"]
df_merg["Age_during_2nd_scan"] = df_merg["Year_scan_imaging_visit"] - df_merg["Birthyear"]
df_merg["Age"] = df_merg["Age_during_scan"]
df_merg.loc[df_merg["Age_during_2nd_scan"] > 0, "Age"] = df_merg["Age_during_2nd_scan"]
second_scans = df_merg[df_merg["Age_during_2nd_scan"] > 0].copy()
second_scans["ScanType"] = "SecondScan"
df_merg = df_merg.drop(second_scans.index)
df_merg["ScanType"] = "FirstScan"
combined_df = pd.concat([df_merg, second_scans], ignore_index=False)

combined_df


In [ ]:
#let's create a large pandas dataframe with all the data we have up to this moment (we still need to figure out how to calculate the age, because something is going wrong
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"
data_1 = os.path.join(csv_path, "aseg_normmean_stats.csv")
data_2 = os.path.join(csv_path, "lh_stats.csv")
data_3 = os.path.join(csv_path, "rh_stats.csv")
data_4 = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
data_5 = os.path.join(csv_path, "Sex_birthyear.csv")

df_1 = pd.read_csv(data_1)
df_2 = pd.read_csv(data_2)
df_3 = pd.read_csv(data_3)
df_4 = pd.read_csv(data_4)
df_5 = pd.read_csv(data_5)

df_1 = df_1.dropna(axis=1)
df_2 = df_2.dropna(axis=1)
df_3 = df_3.dropna(axis=1)
df_4 = df_4.dropna(axis=1)
df_5 = df_5.dropna(axis=1)

df_1 = df_1.reset_index()
df_2 = df_2.reset_index()
df_3 = df_3.reset_index()
df_4 = df_4.reset_index()
df_5 = df_5.reset_index()

#this now merges correctly!

combined_df = df_1.merge(df_2, on="SubjectID", how="left")
combined_df = combined_df.merge(df_3, on="SubjectID", how="left")

#still need to see how to combine the other dataframes --> maybe i need to ask where the correct files are if i cannot locate them myself
#also i should delete the _long and _scan2 files

#combined_df = combined_df.merge(df_4, on="SubjectID", how="left")
#combined_df = combined_df.merge(df_5, on="SubjectID", how="left")
#combined_df = combined_df.set_index("SubjectID")

drop_scans = combined_df[combined_df['SubjectID'].str.endswith('_scan2')].index
combined_df = combined_df.drop(drop_scans)

drop_scans = combined_df[combined_df['SubjectID'].str.endswith('_long')].index
combined_df = combined_df.drop(drop_scans)

combined_df.head(50)
# print(df_1.shape)
# print(df_2.shape)
# print(df_3.shape)


In [ ]:
#for df4
df_4["SubjectID"] = df_4["SubjectID"].astype(int)
combined_df["SubjectID"] = combined_df["SubjectID"].astype(int)

combined_df = combined_df.loc[:, ~combined_df.columns.str.contains("^index", case=False)]
df_4 = df_4.loc[:, ~df_4.columns.str.contains("^index", case=False)]

combined_df = combined_df.merge(df_4, on="SubjectID", how="left")



#for df5
df_5["SubjectID"] = df_5["SubjectID"].astype(int)
combined_df["SubjectID"] = combined_df["SubjectID"].astype(int)

combined_df = combined_df.loc[:, ~combined_df.columns.str.contains("^index", case=False)]
df_5 = df_5.loc[:, ~df_5.columns.str.contains("^index", case=False)]

combined_df_5 = combined_df.merge(df_5, on="SubjectID", how="left")
#combined_df_5.head(50)


pd.set_option('display.max_columns', None)
combined_df_5


In [ ]:
combined_df_5["Year_scan_initial"] = combined_df_5["Year_scan_initial"].fillna(-1)
combined_df_5["Year_scan_initial"] = combined_df_5["Year_scan_initial"].astype(int)

combined_df_5["Birthyear"] = combined_df_5["Birthyear"].fillna(-1)
combined_df_5["Birthyear"] = combined_df_5["Birthyear"].astype(int)


combined_df_5["Age"]  = combined_df_5["Year_scan_initial"] - combined_df_5["Birthyear"]
combined_df_5

In [ ]:
combined_df_5.to_csv("final_datatabel.csv")

In [ ]:
#final clean up of the resulting dataset
df = pd.read_csv("final_datatabel.csv")
df.drop(columns=["Month_scan_initial","Unnamed: 0"],inplace=True)

df.rename(columns={'Year_scan_initial': 'Year_initial_scan', 'Assessment_centre_initial': 'Site'}, inplace=True)
df.set_index("SubjectID")

In [ ]:
df.to_csv("Final_Data.csv")

In [ ]:
csv_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/"

data_4 = os.path.join(csv_path, "Yr_mnth_site_scan.csv")
df_4 = pd.read_csv(data_4)
df_4 = df_4.rename(columns={"Assessment_centre_1_repeat" : "Site_imaging"})
df_5 = df_4[["SubjectID", "Site_imaging"]].copy()
df_5


#let's see if this merges with the other data

In [ ]:
df = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_Data.csv")
df_check = df.merge(df_5, how="left")
df_check = df_check.drop(columns=["Site"])
df_check = df_check.rename(columns={"Site_imaging":"Site"})
#df_check.to_csv("Final_data_corrected_site.csv")
df_check

In [ ]:
health_outcomes = pd.read_csv("/project_cephfs/3022017.06/UKB/phenotypes/current/50_health_outcomes.csv")
health_outcomes.head()